In [ ]:
import pandas as pd
import numpy as np
import utils
import os
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold, ParameterGrid, GridSearchCV
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score, confusion_matrix
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
import importlib
import joblib
import warnings
warnings.filterwarnings('ignore')

%load_ext autoreload
%autoreload 2

In [ ]:
data_folder = 'data/'
raw_train_df = pd.read_csv(data_folder + 'train.csv')
raw_test_df = pd.read_csv(data_folder + 'test.csv')
train_demo_df = pd.read_csv(data_folder + 'train_demographics.csv')
test_demo_df = pd.read_csv(data_folder + 'test_demographics.csv')

temp_calculations_folder_name = 'temp_calculations/'
model_run_folder_name = 'model_runs/'
os.makedirs(temp_calculations_folder_name, exist_ok=True)
os.makedirs(model_run_folder_name, exist_ok=True)

In [ ]:
sampling_rate = 100
n_splits = 3
pipe_name = 'imu_extractor'
model_run_name = 'binary_rf.pkl'
path_to_model_run_name = model_run_folder_name + model_run_name

tof_columns = [f'tof_{i}_v{j}' for i in range(1, 6) for j in range(0, 64)]
some_sequences = ['SEQ_000007', 'SEQ_000008', 'SEQ_000013', 'SEQ_000034', 
                  'SEQ_000046', 'SEQ_000053', 'SEQ_000058', 'SEQ_000063', 'SEQ_000079']

tof_1 = [f'tof_1_v{j}' for j in range(64)]
tof_2 = [f'tof_2_v{j}' for j in range(64)]
tof_3 = [f'tof_3_v{j}' for j in range(64)]
tof_4 = [f'tof_4_v{j}' for j in range(64)]
tof_5 = [f'tof_5_v{j}' for j in range(64)]

linear_edges = np.arange(0, 51, 10)
log_edges    = np.logspace(np.log10(0.5), np.log10(50), num=10)

In [ ]:
# BLOCK 4: Prepare Data with Binary Target
train_df = raw_train_df.set_index('row_id')

# Create binary target (1 = Target, 0 = Non-Target)
train_df['is_target'] = (train_df['sequence_type'] == 'Target').astype(int)

# Sample data (adjust as needed)
train_sample_df, test_sample_df = utils.sample_balanced_split(train_df, train_pct=0.2, test_pct=0.2)

# train_sample_df = train_df[train_df['sequence_id'].isin(some_sequences)]

print(f"Binary target distribution:")
print(train_sample_df.groupby('sequence_id')['is_target'].first().value_counts())


In [ ]:
importlib.reload(utils)

num_pattern = 'acc|rot|thm|tof'
suspect_cols = ['age', 'height_cm', 'shoulder_to_wrist_cm', 'elbow_to_wrist_cm']
cat_cols = ['orientation']
normal_cols = ['adult_child', 'sex', 'handedness']
ordinal_cols = ['segment_id']

sliced_data_df = train_sample_df.copy(deep=True)

preprocessor = ColumnTransformer(
    transformers=[
        ('feature_num_cols', StandardScaler(), make_column_selector(pattern=num_pattern)),
        ('subject_num_cols', StandardScaler(), utils.existing_cols(suspect_cols)),
        ('cat_encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False), 
         utils.existing_cols(cat_cols)),
        ('normal_cols', 'passthrough', utils.existing_cols(normal_cols)),
        ('ordinal_cols', 'passthrough', utils.existing_cols(ordinal_cols))
    ],
    remainder='drop',
    verbose_feature_names_out=False
)
preprocessor.set_output(transform='pandas')

# BLOCK 6: Feature Extractor
custom_extractor = utils.ImuExtractor(subject_df=train_demo_df)

# BLOCK 7: Binary Random Forest
rf_clf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1,
    class_weight='balanced'  # Handle any imbalance
)

pipeline = Pipeline([
    (pipe_name, custom_extractor),
    ('preprocessor', preprocessor),
    ('classifier', utils.ManyToOneWrapper(
        estimator=rf_clf,
        extractor=custom_extractor,
        mode=None,
        target='is_target'
    ))
])

param_grid = {
    f'{pipe_name}__imu_sensor_list':        [['acc_x', 'acc_y', 'acc_z']],
    f'{pipe_name}__imu_domain':             ['acceleration', 'velocity', 'displacement'],
    f'{pipe_name}__combine_imu_axes':       [True],
    f'{pipe_name}__sampling_rate':          [100],

    f'{pipe_name}__rotation_sensor_list':   [['rot_w', 'rot_x', 'rot_y', 'rot_z']],
    f'{pipe_name}__combine_rot_axes':       [True],

    f'{pipe_name}__thermopile_sensor_list': [['thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5']],
    f'{pipe_name}__tof_sensor_list':        [tof_columns], 

    f'{pipe_name}__dc_offset':              [1.5],
    f'{pipe_name}__band_edges':             [log_edges],
    f'{pipe_name}__category_data':          [True],
    f'{pipe_name}__segmentation':           ['window'],

    'classifier__estimator__n_estimators':    [200],
    'classifier__estimator__max_depth':       [5],
    'classifier__estimator__min_samples_split': [5],
}

# BLOCK 9: Grid Search
y = sliced_data_df[['sequence_id', 'is_target']]

grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=GroupKFold(n_splits=n_splits),
    scoring='accuracy',
    verbose=2,
    n_jobs=1,
    return_train_score=True
)

grid_search.fit(sliced_data_df, y, groups=sliced_data_df['sequence_id'])

model = utils.attach_metadata(grid_search)
joblib.dump(model, path_to_model_run_name)

cv_results_df = pd.DataFrame(grid_search.cv_results_)
cv_results_df.to_csv(model_run_folder_name + 'binary_rf_results.csv', index=False)

print(f"Best score: {grid_search.best_score_:.4f}")

In [ ]:
# BLOCK 11: Evaluate on Holdout
best_model = grid_search.best_estimator_

extractor = best_model.named_steps['imu_extractor']
preprocessor = best_model.named_steps['preprocessor']
classifier = best_model.named_steps['classifier']

X_feat = extractor.transform(test_sample_df)
X_proc = preprocessor.transform(X_feat)

y_true = test_sample_df.drop_duplicates('sequence_id').set_index('sequence_id')['is_target']
y_true = y_true.reindex(X_proc.index)

y_pred = pd.Series(classifier.predict(X_proc), index=X_proc.index)

print("\nClassification Report:")
print(classification_report(y_true, y_pred))

print(f"ROC AUC: {roc_auc_score(y_true, y_pred):.4f}")

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix - Binary Classifier')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

In [ ]:
# Feature Importance for Random Forest
best_model = grid_search.best_estimator_

# Get the classifier from pipeline
classifier = best_model.named_steps['classifier'].estimator_

# Get feature names from preprocessor
feature_names = best_model.named_steps['preprocessor'].get_feature_names_out()

# Get importances
importances = classifier.feature_importances_

# Create DataFrame
feat_imp_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values('importance', ascending=False)

feat_imp_df['percentage'] = 100 * (feat_imp_df['importance'] / feat_imp_df['importance'].sum())
feat_imp_df['name'] = feat_imp_df['feature'].str.split('_').str[0]
feat_imp_df = feat_imp_df.sort_values(by='importance', ascending=False).reset_index(drop=True)
core_features_df = feat_imp_df.groupby('name').agg(**{
    'Proportion': ('percentage', 'sum')
}).sort_values(by='Proportion', ascending=False).reset_index()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10), nrows=2, ncols=1)

sns.barplot(data=core_features_df, x='Proportion', y='name', ax=ax[0], orient='h')
ax[0].set_title('Feature Importance')
ax[0].set_xlabel('Proportion (%)')
ax[0].set_ylabel('Feature Group')

sns.barplot(data=feat_imp_df.head(20), x='percentage', y='feature', ax=ax[1], orient='h')
ax[1].set_title('Feature Importance top 20')
ax[1].set_xlabel('Proportion (%)')
ax[1].set_ylabel('Feature')